# 04. DATA PREPARATION & DATA LEAKAGE MITIGATION
**Chuẩn bị ma trận đặc trưng, phân chia phân tầng và kiểm soát rủi ro rò rỉ dữ liệu (V12 - Duration)**
---
Nội dung notebook này thực hiện:
1. Phân tích chuyên sâu về rủi ro Data Leakage của biến `duration` (thời lượng cuộc gọi chưa biết trước khi gọi).
2. Thiết kế 2 kịch bản thực nghiệm:
   - Kịch bản A: Dự đoán thực tế trước cuộc gọi (Realistic Pre-Call - Loại bỏ duration).
   - Kịch bản B: Chuẩn đối sánh sau cuộc gọi (Post-Call Benchmark - Có duration).
3. One-Hot Encoding và chuẩn hóa StandardScaler chỉ fit trên tập huấn luyện (X_train).


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

df_clean = pd.read_csv('../dataset/cleaned/bank_cleaned.csv')

# Phân chia Stratified 80/20 Holdout
train_df, test_df = train_test_split(df_clean, test_size=0.20, random_state=42, stratify=df_clean['target'])
print(f'Huấn luyện: {len(train_df):,} mẫu | Kiểm tra: {len(test_df):,} mẫu')

# Kịch bản B (Benchmark có duration)
numeric_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
cat_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

X_train_raw = train_df[numeric_cols + cat_cols]
X_test_raw = test_df[numeric_cols + cat_cols]

X_train_enc = pd.get_dummies(X_train_raw, drop_first=True, dtype=int)
X_test_enc = pd.get_dummies(X_test_raw, drop_first=True, dtype=int).reindex(columns=X_train_enc.columns, fill_value=0)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_enc)
X_test_scaled = scaler.transform(X_test_enc)
print('Ma trận đặc trưng đã chuẩn hóa:', X_train_scaled.shape)
